# JSONStorm — Experiment Visualisations

Covers:
1. API latency distribution by prompt
2. Token usage by prompt and query type
3. Generation success rate and retry behaviour
4. Latency vs output tokens scatter
5. Wall time distribution by complexity class
6. Operator count averages per complexity class (SQLStorm Table 6 equivalent)
7. Complexity distribution by prompt
8. Execution success / timeout / error rates by prompt
9. Runtime distribution across all queries (SQLStorm Figure 4 equivalent)
10. Output tokens vs wall time scatter (generation cost to execution cost)

In [1]:
import yanex.results as yr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

plt.rcParams.update({
    'figure.dpi':        150,
    'figure.facecolor':  'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'font.size':         11,
})

COMPLEXITY_COLOURS = {'low': '#4C9BE8', 'medium': '#F5A623', 'high': '#E85454'}
COMPLEXITY_NUM_MAP = {1: 'low', 2: 'medium', 3: 'high'}

In [4]:
import yanex.results as yr

all_metrics = yr.get_metrics(name="gemini-25-flash-lite-prompt-p3")
print(f'Loaded {len(all_metrics)} rows from {all_metrics["experiment_id"].nunique()} experiments')
all_metrics.head()

Loaded 132 rows from 1 experiments


,experiment_id,step,metric_name,value
0,a39cec86,0,param_model,NaN
1,a39cec86,0,param_prompt,NaN
2,a39cec86,0,param_query_type,NaN
3,a39cec86,0,param_num_queries,1.000
4,a39cec86,1,api_latency_ms,3775.226


In [5]:
print('All columns:')
print(list(all_metrics.columns))
print()

All columns:
['experiment_id', 'step', 'metric_name', 'value']



In [6]:
# Adjust to match your experiment naming convention
EXPERIMENT_PATTERN = "gemini-25-flash-lite-prompt-p3"

all_metrics = yr.get_metrics(name=EXPERIMENT_PATTERN)
print(f'Loaded {len(all_metrics)} rows from {all_metrics["metric_name"].nunique()} experiments')
print(f'Columns: {list(all_metrics.columns)}')

Loaded 132 rows from 45 experiments
Columns: ['experiment_id', 'step', 'metric_name', 'value']


# ── Pivot from long to wide format ───────────────────────────────────────────

In [7]:
# Raw shape: one row per (experiment_id, step, metric_name)
# We need: one row per (experiment_id, step) with each metric as its own column

all_metrics_wide = all_metrics.pivot_table(
    index=['experiment_id', 'step'],
    columns='metric_name',
    values='value',
    aggfunc='first',   # each (experiment_id, step, metric_name) should be unique
).reset_index()

# Flatten the column index (pivot_table gives a named MultiIndex)
all_metrics_wide.columns.name = None

print(f'Wide shape: {all_metrics_wide.shape}')
print(f'Columns: {list(all_metrics_wide.columns)}')
all_metrics_wide.head(3)

Wide shape: (12, 40)
Columns: ['experiment_id', 'step', 'api_attempt', 'api_input_tokens', 'api_latency_ms', 'api_output_tokens', 'api_success', 'api_total_tokens', 'api_validation_fail', 'avg_docs_examined', 'avg_keys_examined', 'avg_wall_time_ms', 'docs_examined', 'error', 'execution_complexity_num', 'is_mismatch', 'keys_examined', 'max_depth', 'max_wall_time_ms', 'n_computed_fields', 'n_facet', 'n_group', 'n_lookup', 'n_or_clauses', 'n_predicates', 'n_returned', 'n_sort', 'n_unwind', 'n_window', 'param_num_queries', 'param_timeout_ms', 'queries_error', 'queries_success', 'queries_timeout', 'queries_total', 'sample_n', 'timeout', 'timeout_ms', 'total_operators', 'wall_time_ms']


,experiment_id,step,api_attempt,api_input_tokens,api_latency_ms,api_output_tokens,api_success,api_total_tokens,api_validation_fail,avg_docs_examined,...,param_timeout_ms,queries_error,queries_success,queries_timeout,queries_total,sample_n,timeout,timeout_ms,total_operators,wall_time_ms
0,a39cec86,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,a39cec86,1,1.0,2309.0,3775.226,141.0,1.0,2450.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,53.498
2,a39cec86,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,42.611


In [8]:
# Generation rows — have api_latency_ms and a step value
gen_df = all_metrics_wide[all_metrics_wide['api_latency_ms'].notna()].copy()

# Harness per-query rows — have wall_time_ms AND a step value
harness_query_df = all_metrics_wide[
    all_metrics_wide['wall_time_ms'].notna() & all_metrics_wide['step'].notna()
].copy()

# Harness summary rows — logged once per run without a step
harness_summary_df = all_metrics_wide[
    all_metrics_wide['success_rate'].notna() & all_metrics_wide['step'].isna()
].copy()

# Add human-readable complexity label to per-query rows
if 'execution_complexity_num' in harness_query_df.columns:
    harness_query_df['complexity'] = (
        harness_query_df['execution_complexity_num']
        .map(COMPLEXITY_NUM_MAP)
        .fillna('unknown')
    )

# Also pull param_prompt and param_query_type into harness rows if they
# only appear in the summary row for that experiment — merge them in.
if 'param_prompt' not in harness_query_df.columns or harness_query_df['param_prompt'].isna().all():
    param_cols = [c for c in harness_summary_df.columns
                  if c.startswith('param_') and c in harness_summary_df.columns]
    if param_cols:
        params = (
            harness_summary_df[['experiment_id'] + param_cols]
            .dropna(subset=['experiment_id'])
            .drop_duplicates('experiment_id')
        )
        harness_query_df = harness_query_df.merge(params, on='experiment_id', how='left')
        gen_df = gen_df.merge(params, on='experiment_id', how='left')

print(f'Generation rows      : {len(gen_df)}')
print(f'Harness per-query    : {len(harness_query_df)}')
print(f'Harness summary rows : {len(harness_summary_df)}')

KeyError: 'success_rate'

## 1. API latency distribution by prompt

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

prompts = sorted(gen_df['param_prompt'].dropna().unique())
latency_data = [
    gen_df.loc[gen_df['param_prompt'] == p, 'api_latency_ms'].dropna().values
    for p in prompts
]

bp = ax.boxplot(
    latency_data, labels=prompts, patch_artist=True,
    medianprops=dict(color='black', linewidth=1.5),
    flierprops=dict(marker='o', markersize=3, alpha=0.4),
)
colours = plt.cm.tab10(np.linspace(0, 0.7, len(prompts)))
for patch, colour in zip(bp['boxes'], colours):
    patch.set_facecolor(colour)
    patch.set_alpha(0.7)

ax.set_xlabel('Prompt')
ax.set_ylabel('API latency (ms)')
ax.set_title('Gemini API latency distribution by prompt')
plt.tight_layout()
plt.show()

## 2. Token usage by prompt and query type

In [ ]:
token_agg = (
    gen_df
    .groupby(['param_prompt', 'param_query_type'])[['api_input_tokens', 'api_output_tokens']]
    .mean()
    .reset_index()
)

query_types = sorted(token_agg['param_query_type'].dropna().unique())
prompts_ord = sorted(token_agg['param_prompt'].unique())
x = np.arange(len(prompts_ord))
width = 0.35 / max(len(query_types), 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, title in zip(
    axes,
    ['api_input_tokens', 'api_output_tokens'],
    ['Avg input tokens (prompt size)', 'Avg output tokens (query length)'],
):
    for i, qt in enumerate(query_types):
        subset = token_agg[token_agg['param_query_type'] == qt]
        vals = [
            subset.loc[subset['param_prompt'] == p, col].values[0]
            if p in subset['param_prompt'].values else 0
            for p in prompts_ord
        ]
        ax.bar(x + i * width, vals, width, label=qt, alpha=0.8)
    ax.set_xticks(x + width * (len(query_types) - 1) / 2)
    ax.set_xticklabels(prompts_ord)
    ax.set_xlabel('Prompt')
    ax.set_ylabel('Tokens')
    ax.set_title(title)
    ax.legend(title='Query type')

fig.suptitle('Token usage by prompt and query type', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Generation success rate and retry behaviour

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Stacked bar - outcome shares per prompt
outcome_agg = (
    gen_df
    .assign(
        outcome=lambda d: np.select(
            [d['api_success'] == 1, d['api_validation_fail'] == 1],
            ['success',            'validation fail'],
            default='other fail',
        )
    )
    .groupby(['param_prompt', 'outcome'])
    .size()
    .unstack(fill_value=0)
)
outcome_pct = outcome_agg.div(outcome_agg.sum(axis=1), axis=0) * 100
colour_outcome = {'success': '#4CAF50', 'validation fail': '#F5A623', 'other fail': '#E85454'}
bottom = np.zeros(len(outcome_pct))
for col in ['success', 'validation fail', 'other fail']:
    if col in outcome_pct.columns:
        axes[0].bar(
            outcome_pct.index, outcome_pct[col], bottom=bottom,
            label=col, color=colour_outcome[col], alpha=0.85,
        )
        bottom += outcome_pct[col].values
axes[0].set_xlabel('Prompt')
axes[0].set_ylabel('Share of API calls (%)')
axes[0].set_title('Generation outcome by prompt')
axes[0].legend()

# Mean attempt number per prompt
attempt_agg = gen_df.groupby('param_prompt')['api_attempt'].mean().sort_index()
attempt_agg.plot(kind='bar', ax=axes[1], color='#4C9BE8', alpha=0.8, width=0.6)
axes[1].axhline(1, color='black', linewidth=0.8, linestyle='--', label='no retries')
axes[1].set_xlabel('Prompt')
axes[1].set_ylabel('Mean attempt number')
axes[1].set_title('Average retries needed per prompt')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. API latency vs output tokens scatter

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for qt, marker in [('find', 'o'), ('aggregate', 's')]:
    subset = gen_df[gen_df['param_query_type'] == qt]
    ax.scatter(
        subset['api_output_tokens'], subset['api_latency_ms'],
        marker=marker, alpha=0.4, s=20, label=qt,
    )

valid = gen_df[gen_df['api_success'] == 1][['api_output_tokens', 'api_latency_ms']].dropna()
if len(valid) > 2:
    z = np.polyfit(valid['api_output_tokens'], valid['api_latency_ms'], 1)
    xs = np.linspace(valid['api_output_tokens'].min(), valid['api_output_tokens'].max(), 200)
    ax.plot(xs, np.poly1d(z)(xs), 'k--', linewidth=1.2, label='trend (successful calls)')

ax.set_xlabel('Output tokens')
ax.set_ylabel('API latency (ms)')
ax.set_title('API latency vs output tokens')
ax.legend(title='Query type')
plt.tight_layout()
plt.show()

## 5. Wall time distribution by complexity class

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

classes = ['low', 'medium', 'high']
wt_data = [
    harness_query_df.loc[harness_query_df['complexity'] == c, 'wall_time_ms'].dropna().values
    for c in classes
]
labels = [f'{c}\n(n={len(d)})' for c, d in zip(classes, wt_data)]

bp = ax.boxplot(
    wt_data, labels=labels, patch_artist=True,
    medianprops=dict(color='black', linewidth=1.5),
    flierprops=dict(marker='o', markersize=3, alpha=0.35),
)
for patch, cls in zip(bp['boxes'], classes):
    patch.set_facecolor(COMPLEXITY_COLOURS[cls])
    patch.set_alpha(0.75)

ax.set_xlabel('Complexity class')
ax.set_ylabel('Wall time (ms)')
ax.set_title('Query execution time by complexity class')
plt.tight_layout()
plt.show()

for cls, data in zip(classes, wt_data):
    if len(data):
        print(f'  {cls:6s}  median={np.median(data):.1f} ms  '
              f'p95={np.percentile(data, 95):.1f} ms  n={len(data)}')

## 6. Operator count averages per complexity class (SQLStorm Table 6 equivalent)

In [ ]:
operator_features = [
    'total_operators', 'n_lookup', 'n_group', 'n_unwind',
    'n_sort', 'n_window', 'n_facet', 'n_predicates',
    'n_computed_fields', 'max_depth',
]

has_summary_avgs = (
    not harness_summary_df.empty
    and any(f'avg_{f}_low' in harness_summary_df.columns for f in operator_features)
)

if has_summary_avgs:
    # Use the pre-computed per-class averages from compute_summary_metrics
    table6 = pd.DataFrame(
        {
            cls: [
                harness_summary_df[f'avg_{feat}_{cls}'].mean()
                if f'avg_{feat}_{cls}' in harness_summary_df.columns else 0
                for feat in operator_features
            ]
            for cls in ['low', 'medium', 'high']
        },
        index=operator_features,
    )
    print('Source: summary rows (avg_{feature}_{class})')
else:
    # Fall back to computing from per-query rows
    available = [f for f in operator_features if f in harness_query_df.columns]
    table6 = (
        harness_query_df[harness_query_df['complexity'].isin(['low', 'medium', 'high'])]
        .groupby('complexity')[available]
        .mean().T
        .reindex(columns=['low', 'medium', 'high'])
    )
    print('Source: per-query rows (fallback)')

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(table6))
width = 0.25
for i, cls in enumerate(['low', 'medium', 'high']):
    if cls in table6.columns:
        ax.bar(x + i * width, table6[cls], width, label=cls,
               color=COMPLEXITY_COLOURS[cls], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(table6.index, rotation=35, ha='right')
ax.set_ylabel('Average count per query')
ax.set_title('Average operator counts by complexity class (SQLStorm Table 6 equivalent)')
ax.legend(title='Complexity')
plt.tight_layout()
plt.show()

display(table6.round(2))

## 7. Complexity distribution by prompt

In [ ]:
has_summary_counts = (
    not harness_summary_df.empty
    and 'n_complexity_low' in harness_summary_df.columns
    and 'param_prompt' in harness_summary_df.columns
)

if has_summary_counts:
    comp_dist = (
        harness_summary_df
        .groupby('param_prompt')[['n_complexity_low', 'n_complexity_medium', 'n_complexity_high']]
        .sum()
        .rename(columns={'n_complexity_low': 'low',
                         'n_complexity_medium': 'medium',
                         'n_complexity_high': 'high'})
    )
else:
    comp_dist = (
        harness_query_df[harness_query_df['complexity'].isin(['low', 'medium', 'high'])]
        .groupby(['param_prompt', 'complexity'])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=['low', 'medium', 'high'], fill_value=0)
    )

comp_pct = comp_dist.div(comp_dist.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Percentage stacked bar
bottom = np.zeros(len(comp_pct))
for cls in ['low', 'medium', 'high']:
    if cls in comp_pct.columns:
        axes[0].bar(comp_pct.index, comp_pct[cls], bottom=bottom,
                    label=cls, color=COMPLEXITY_COLOURS[cls], alpha=0.85)
        bottom += comp_pct[cls].values
axes[0].set_xlabel('Prompt')
axes[0].set_ylabel('Share of queries (%)')
axes[0].set_title('Complexity distribution by prompt (% share)')
axes[0].legend(title='Complexity')

# Absolute count grouped bar
x = np.arange(len(comp_dist))
width = 0.25
for i, cls in enumerate(['low', 'medium', 'high']):
    if cls in comp_dist.columns:
        axes[1].bar(x + i * width, comp_dist[cls], width,
                    label=cls, color=COMPLEXITY_COLOURS[cls], alpha=0.85)
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(comp_dist.index)
axes[1].set_xlabel('Prompt')
axes[1].set_ylabel('Number of queries')
axes[1].set_title('Complexity distribution by prompt (counts)')
axes[1].legend(title='Complexity')

plt.tight_layout()
plt.show()

## 8. Execution success / timeout / error rates by prompt

In [ ]:
has_rate_cols = (
    not harness_summary_df.empty
    and 'success_rate' in harness_summary_df.columns
    and 'param_prompt' in harness_summary_df.columns
)

if has_rate_cols:
    rate_df = (
        harness_summary_df
        .groupby('param_prompt')[['success_rate', 'timeout_rate', 'error_rate']]
        .mean()
        .rename(columns={'success_rate': 'success',
                         'timeout_rate': 'timeout',
                         'error_rate':   'error'})
    ) * 100
else:
    rate_df = (
        harness_query_df
        .assign(success=lambda d: d['wall_time_ms'].notna().astype(int))
        .groupby('param_prompt')[['success']]
        .mean() * 100
    )

fig, ax = plt.subplots(figsize=(9, 5))
colour_rate = {'success': '#4CAF50', 'timeout': '#F5A623', 'error': '#E85454'}
bottom = np.zeros(len(rate_df))
for col in ['success', 'timeout', 'error']:
    if col in rate_df.columns:
        ax.bar(rate_df.index, rate_df[col], bottom=bottom,
               label=col, color=colour_rate[col], alpha=0.85)
        bottom += rate_df[col].values

ax.set_xlabel('Prompt')
ax.set_ylabel('Share of queries (%)')
ax.set_title('Execution outcome by prompt')
ax.legend(title='Outcome')
plt.tight_layout()
plt.show()

display(rate_df.round(1))

## 9. Runtime distribution across all queries (SQLStorm Figure 4 equivalent)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

wt_positive = harness_query_df['wall_time_ms'][harness_query_df['wall_time_ms'] > 0]

if len(wt_positive) == 0:
    print('No wall_time_ms data found.')
else:
    log_bins = np.logspace(
        np.log10(wt_positive.min()),
        np.log10(wt_positive.max()),
        55,
    )
    prompts_with_data = (
        harness_query_df
        .dropna(subset=['wall_time_ms', 'param_prompt'])
        ['param_prompt'].unique()
    )
    colours_p = plt.cm.tab10(np.linspace(0, 0.7, len(prompts_with_data)))

    for p, colour in zip(sorted(prompts_with_data), colours_p):
        vals = harness_query_df.loc[
            (harness_query_df['param_prompt'] == p) &
            (harness_query_df['wall_time_ms'] > 0),
            'wall_time_ms'
        ].dropna().values
        if len(vals) < 5:
            continue
        counts, edges = np.histogram(vals, bins=log_bins, density=True)
        centres = (edges[:-1] + edges[1:]) / 2
        ax.plot(centres, counts, label=p, color=colour, linewidth=1.5, alpha=0.85)

    ax.set_xscale('log')
    ax.set_xlabel('Wall time (ms, log scale)')
    ax.set_ylabel('Density')
    ax.set_title('Runtime distribution by prompt (SQLStorm Figure 4 equivalent)')
    ax.legend(title='Prompt')
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))

plt.tight_layout()
plt.show()

## 10. Output tokens vs wall time (generation cost to execution cost)

In [ ]:
can_join = (
    'step' in gen_df.columns and
    'step' in harness_query_df.columns and
    'name' in gen_df.columns and
    'name' in harness_query_df.columns
)

if can_join:
    joined = pd.merge(
        gen_df[['name', 'step', 'api_output_tokens', 'param_query_type']].dropna(),
        harness_query_df[['name', 'step', 'wall_time_ms', 'complexity']].dropna(),
        on=['name', 'step'],
        how='inner',
    )
    print(f'Joined {len(joined)} rows')

    fig, ax = plt.subplots(figsize=(9, 5))
    for cls in ['low', 'medium', 'high']:
        subset = joined[joined['complexity'] == cls]
        ax.scatter(
            subset['api_output_tokens'], subset['wall_time_ms'],
            c=COMPLEXITY_COLOURS[cls], label=cls, alpha=0.4, s=18,
        )

    ax.set_xlabel('Output tokens (generation cost)')
    ax.set_ylabel('Wall time ms (execution cost)')
    ax.set_title('Generation length vs execution time\nColour = complexity class')
    ax.legend(title='Complexity')
    plt.tight_layout()
    plt.show()
else:
    print('Cannot join: need step and name columns in both DataFrames.')
    print('Run both scripts via yanex run so step values are recorded.')